In [4]:
# Turns requirements/*.yaml into CSVs for Supabase.
# Requirement fields (only name is required):
#   min_credits             credits needed
#   max_credits             most credits that can count
#   min_children            children needed (default: all)
#   min_essential_children  how many of those must be essential
#   courses                 courses that count
#   match                   rule that finds more courses in courses.csv

import glob
import re

import pandas as pd
import yaml

# all offered courses
courses_df = pd.read_csv("tables/courses.csv")
# one per program
programs = [yaml.safe_load(open(f)) for f in sorted(glob.glob("requirements/*.yaml"))]


def slug(name):
    # "CS Breadth" -> "cs-breadth"
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def expand_match(match, df):
    # start with all courses, then filter by each rule
    sel = pd.Series(True, index=df.index)
    if "subject_contains" in match:
        # by subject name, so cross-listed courses count
        sel &= df["subject"].str.contains(match["subject_contains"], case=False, na=False)
    if "academic_level" in match:
        # grad or undergrad
        sel &= df["academic_level"] == match["academic_level"]
    if "code_prefix" in match:
        # e.g. "RBE"
        sel &= df["course_code"].str.startswith(match["code_prefix"] + " ")
    if "code_min" in match:
        # "CS 411X" -> 411
        sel &= pd.to_numeric(df["course_code"].str.extract(r"(\d+)")[0], errors="coerce") >= match["code_min"]
    # drop excluded courses
    return sorted(set(df.loc[sel, "course_code"]) - set(match.get("exclude", [])))

In [5]:
# rows for the three tables
program_rows, requirement_rows, link_rows = [], [], []


def walk(node, program_id, parent_id, req_id):
    # save this requirement, then its children
    # id is a path, e.g. "cs-bsms/cs-breadth/theory"
    children = node.get("children", [])
    requirement_rows.append(
        {
            "id": req_id,
            "program_id": program_id,
            "parent_id": parent_id,
            "name": node["name"],
            "min_credits": node.get("min_credits"),
            "max_credits": node.get("max_credits"),
            # default: all children
            "min_children": node.get("min_children", len(children) or None),
            "min_essential_children": node.get("min_essential_children"),
            "is_essential": node.get("essential", False),
        }
    )
    # listed courses + matched courses
    codes = node.get("courses", [])
    if "match" in node:
        codes = codes + expand_match(node["match"], courses_df)
    link_rows.extend({"requirement_id": req_id, "course_code": c} for c in codes)
    for child in children:
        walk(child, program_id, req_id, f"{req_id}/{slug(child['name'])}")


for p in programs:
    program_rows.append({k: p.get(k) for k in ["id", "name", "catalog_url", "total_credits"]})
    for req in p["requirements"]:
        # top level has no parent
        walk(req, p["id"], None, f"{p['id']}/{slug(req['name'])}")

programs_df = pd.DataFrame(program_rows)
requirements_df = pd.DataFrame(requirement_rows)
# remove duplicate links
links_df = pd.DataFrame(link_rows).drop_duplicates()

In [6]:
# write the CSVs
programs_df.to_csv("tables/programs.csv", index=False)
requirements_df.to_csv("tables/requirements.csv", index=False)
links_df.to_csv("tables/requirement_courses.csv", index=False)